In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split, KFold, cross_val_score, learning_curve
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

# 1. Load the data
df = pd.read_csv('house_price_data.csv')

# 2. Impute missing values (crucial for small datasets)
df['Lot Area'] = df['Lot Area'].fillna(df['Lot Area'].median())
df['Floor Area'] = df['Floor Area'].fillna(df['Floor Area'].median())

# 3. Filter Outliers (keeps data stable)
q_low, q_hi = df["Price"].quantile([0.05, 0.95])
df = df[(df["Price"] < q_hi) & (df["Price"] > q_low)].copy()

# 4. Prepare Features
df_encoded = pd.get_dummies(df, columns=['Barangay'], drop_first=True)
X = df_encoded.drop('Price', axis=1)
y = df_encoded['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

FileNotFoundError: [Errno 2] No such file or directory: 'house_price_data.xlsx - Sheet1.csv'

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=150,
    max_depth=2,          
    learning_rate=0.05,
    reg_lambda=15,        # High regularization to prevent overfitting
    subsample=0.8,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
# 1. Mean Absolute Percentage Error (MAPE)
y_pred = model.predict(X_test)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

# 2. K-Fold Cross Validation (Stability Check)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = -cross_val_score(model, X, y, cv=kf, scoring='neg_mean_absolute_error')

print(f"Average Error (MAE): PHP {cv_scores.mean():,.2f}")
print(f"Accuracy Error: {mape:.2f}% (Average percentage the model is off)")

# 3. Visualizations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Residual Plot (Check for Bias)
ax1.scatter(y_pred, y_test - y_pred, alpha=0.5, color='orange')
ax1.axhline(y=0, color='r', linestyle='--')
ax1.set_title('Residuals (Should be randomly scattered)')

# Learning Curve
train_sizes, train_scores, test_scores = learning_curve(
    model, X, y, cv=kf, scoring='neg_mean_absolute_error', train_sizes=np.linspace(0.1, 1.0, 10)
)
ax2.plot(train_sizes, -np.mean(train_scores, axis=1), label='Train Error')
ax2.plot(train_sizes, -np.mean(test_scores, axis=1), label='Validation Error')
ax2.set_title('Learning Curve (Bias/Variance Check)')
ax2.legend()
plt.show()

In [ ]:
def analyze_barangay(barangay_name, annual_growth=0.06):
    barangay_name = barangay_name.upper()
    b_data = df[df['Barangay'] == barangay_name]
    
    if len(b_data) == 0:
        return f"Error: {barangay_name} not found in the dataset."
    
    current_avg = b_data['Price'].mean()
    town_avg = df['Price'].mean()
    
    # Value Assessment
    status = "GOOD VALUE (Below Market)" if current_avg < town_avg else "PREMIUM (Above Market)"
    
    # Years to project
    projection_years = [0, 5, 10, 25]
    results = {yr: current_avg * ((1 + annual_growth) ** yr) for yr in projection_years}
    
    print(f"--- REPT FOR BARANGAY: {barangay_name} ---")
    print(f"Sample Size: {len(b_data)} listings")
    print(f"Assessment: {status}")
    print(f"Current Avg: PHP {current_avg:,.2f}")
    print("-" * 35)
    print("FUTURE PRICE PROJECTIONS (assuming 6% annual growth):")
    for yr, val in results.items():
        print(f" In {yr:2d} Years: PHP {val:,.2f}")

# Usage:
analyze_barangay('JULUGAN')